**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Chapter 6: Ken French Portfolio Returns with `farms`

In this notebook, we move from simulated returns to observed historical returns. The name `farms` refers to **Financial Analysis and Risk Management**, the Python package used in this course. We use its `load_ken_french_data()` function to download Ken French strategy portfolios from the [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html). We begin with monthly size deciles, then compare deciles, quintiles, strategy types, and available portfolio frequencies before estimating summary statistics and plotting return histories.

## Complete the preparation notebooks first

Before starting this data lab, work through these two short Chapter 6 notebooks:

1. `Ex06-pandas_intro.ipynb`
2. `Ex06-matplotlib_intro.ipynb`

These notebooks use small, made-up return data so you can learn the tools without waiting for a download. The Pandas notebook teaches you how to inspect a DataFrame, select portfolio columns, filter observations, create new columns, and calculate summary statistics. The Matplotlib notebook teaches you how to create labeled time-series plots, histograms, comparison charts, and cumulative-wealth plots.

This lab combines both skills with historical data. The `farms` function returns a Pandas DataFrame, and the later cells ask you to summarize and visualize the portfolio returns. After completing the preparation notebooks, you should be comfortable reading code such as `portfolio_returns.head()`, selecting columns such as `Dec 1` and `Dec 10`, and working with `fig` and `ax` to make a plot.

## Learning objectives

By the end of this notebook, you should be able to:

- Install and import the `farms` package.
- Explain how portfolio sorts group firms by a characteristic.
- Download monthly decile and quintile portfolios with `farms`.
- Use `load_ken_french_data()` with different data types, strategies, portfolio selections, and frequencies.
- Explain which strategies have published daily portfolio files.
- Estimate historical mean return and volatility for strategy portfolios.
- Annualize monthly estimates.
- Calculate portfolio excess returns and compare annualized Sharpe ratios across momentum deciles.
- Plot a return distribution and the growth of one dollar invested.
- Explain why historical summary statistics are sample estimates rather than known population parameters.

## What is a portfolio sort?

A portfolio sort ranks firms by a characteristic, divides them into groups, and calculates the return on each group. For example, the Ken French size decile portfolios sort firms by market equity into 10 portfolios: `Dec 1` is the portfolio that contains the firms with the lowest market equity, while `Dec 10` contains the firms with the highest market equity. The portfolios are eithier value- or equal-weighted, depending on which option you choose.

The sort is formed using the breakpoints and timing specified by the Ken French data library. A breakpoint is a cutoff value for the sorting characteristic that determines which portfolio a stock belongs to. For example, when forming size deciles, Ken French calculates the 10th, 20th, through 90th percentiles of market capitalization using all NYSE firms. Firms below the 10th-percentile cutoff are assigned to `Dec 1` firms between the 10th and 20th percentiles to `Dec 2`, and so forth. These breakpoints are then applied to all eligible stocks, not just NYSE stocks, and are recalculated according to the library’s specified schedule, such as annually in June or monthly.

## Imports and setup

We install `farms` and import the packages used in the analysis. `farms` includes Matplotlib and the plotting helpers used by this lab's charts.

In [ ]:
%pip install -q "farms"

import farms as fm
import farms.plotting as fp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Keep displayed tables compact and readable.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Available portfolio sorts



The current package supports the following portfolio sorts:

| Sort name | Characteristic used to form portfolios |
| --- | --- |
| `accruals` | Operating working-capital accruals |
| `beta` | Historical market beta |
| `booktomarket` | Book equity relative to market equity (BE/ME) |
| `dividendyield` | Dividend yield (D/P) |
| `earningsprice` | Earnings relative to price (E/P) |
| `idiosyncraticvariance` | Residual-return variance from the Fama--French three-factor model |
| `investment` | Change in total assets relative to lagged total assets |
| `momentum` | Prior returns from months t-12 through t-2 |
| `netissuances` | Net share issuance |
| `profitability` | Operating profitability |
| `shorttermreversal` | Prior one-month return |
| `size` | Market equity |
| `variance` | Variance of daily returns |


## Accessing Portfolio-Sorted Returns
The `load_ken_french_data()` function provides access to a variety of historical return data posted on his website. Here we describe how to access returns for portfolios sorted on firm characteristics described above.  The inputs below allow you to specify the portfolio grouping, sorting strategy, weighting method, frequency, sample period, and particular portfolios to retrieve.

`load_ken_french_data()`

**Required inputs:**
- `data_type`: Choose the portfolio granularity:
  - `deciles` divide stocks into 10 portfolios, labeled `Dec 1` through `Dec 10`.
  - `quintiles` divide stocks into 5 portfolios, labeled `Qnt 1` through `Qnt 5`.
- `strategy`: Select the characteristic used to sort stocks.  Available strategies are listed above.

**Optional inputs:**

- `weighting`: Choose `value` or `equal` weighting; defaults to `value`.
- `portfolio`: Select all portfolios or specific groups; defaults to `all`.
  - `all`: Return every portfolio.
  - An integer: Return one portfolio, such as `portfolio=1`.
  - A list or tuple of integers: Return selected portfolios, such as `portfolio=[1, 5, 10]`.
  - For deciles, valid portfolio numbers are 1 through 10. For quintiles, valid portfolio numbers are 1 through 5.
  - `'low'`: Return the lowest-ranked portfolio.
  - `'high'`: Return the highest-ranked portfolio.
- `frequency`: Request monthly or daily data when published for the selected strategy; defaults to `monthly`.
- `start_date` and `end_date`: Define the sample period; defaults to the full available range.
- `include_factors`: Add factor returns to the portfolio DataFrame; defaults to `None`. Use `None` to return only the portfolio columns, `market` to add `mkt-rf` and `rf`, or `ff3`/`ff5` to add the corresponding Fama--French factors.
- `details`: Display detailed information about the selected source and request; defaults to `False`.

Note that not all strategy/data_type/frequency combinations are available. The `load_ken_french_data()` function will raise an error if you request a combination that is not available.  The bottom of the error message explains the invalid combination and lists the available options.  The `available` DataFrame below shows all available combinations of strategy, data_type, and frequency.

In [ ]:
# Example: List the available data for size portfolios.
available = fm.list_ken_french_data()
available[available["strategy"] == "size"]

## Examples

In [ ]:
# Example 1: Load all available monthly value-weighted momentum decile portfolio returns and market factors.
momentum_deciles = fm.load_ken_french_data(
    data_type='deciles',
    strategy='momentum',
    include_factors='market'
)
momentum_deciles.head()

In [ ]:
# Example 2: Load the daily size decile portfolios from January 1990 to the most recent available month. The `details=True` option prints information about the source file, including the date range and portfolio construction. Here we specify a start_date but no end_date.  The function will default to the latest available end_date.
start_date = '1990-01'

daily_size_portfolios = fm.load_ken_french_data(
    data_type='deciles',
    strategy='size',
    start_date=start_date,
    frequency='daily',
    details=True
)

daily_size_portfolios.head()

In [ ]:
# Example 3: Load all available monthly equal book-to-market quintile returns for portfolios 1 and 5
bm_portfolios = fm.load_ken_french_data(
    data_type='quintiles',
    strategy='booktomarket',
    weighting='equal',
    portfolio=[1, 5]
)
bm_portfolios.head()

## Estimate summary statistics and Sharpe ratios

Below we compute summary statistics for the monthly momentum-decile portfolios. The `include_factors='market'` option adds the Fama--French market excess return (`mkt-rf`) and risk-free return (`rf`) to the downloaded portfolio data. The `fm.summary_stats()` function uses the `rf` series to calculate excess returns and uses `frequency='monthly'` to annualize the arithmetic mean, volatility, excess mean, and Sharpe ratio. The function also reports the number of observations for each portfolio. It does not include minimums or percentiles because those are period-specific distribution statistics rather than quantities that should be annualized by simple multiplication.

In [ ]:
# Define the columns containing decile portfolio returns.
portfolio_columns = [column for column in momentum_deciles.columns if column.startswith('Dec ')]
portfolio_returns = momentum_deciles[portfolio_columns]

# Let farms calculate observations and annualized return, risk, and Sharpe metrics.
summary_stats = fm.summary_stats(
    returns=portfolio_returns,
    risk_free=momentum_deciles['rf'],
    frequency='monthly',
)

summary_stats

The `fm.summary_stats()` function uses the observation frequency to annualize the arithmetic mean and volatility. For monthly data, it multiplies the mean by 12 and volatility by the square root of 12. The annualized arithmetic mean is an estimate of the average one-month return expressed at an annual scale; it is not the same as the compound annual growth rate. The annualized excess mean is based on the portfolio return minus `rf`, and the annualized Sharpe ratio is annualized excess mean divided by annualized volatility. `Dec 1` is the low-momentum portfolio and `Dec 10` is the high-momentum portfolio.

## Plot the monthly return distribution

Histograms help us see the range and shape of realized monthly returns. The `fp.plot_return_histograms()` function creates a panel for each selected portfolio, adds a mean-return line by default, and uses a common y-axis so the panels can be compared. We compare the low-momentum, middle-momentum, and high-momentum portfolios with the probability distributions and simulated samples from the earlier Chapter 6 notebooks.

`plot_return_histograms()`

**Required Inputs**

- `returns`: A Pandas DataFrame containing the portfolio returns as decimal values.

**Optional inputs**

- `columns`: Example: `columns=['Dec 1', 'Dec 10']` plots only the low- and high-ranked portfolios; default=all columns.
- `bins`: Example: `bins=15` uses exactly 15 bins; default='auto' estimates a suitable number of bins from the pooled returns.
- `show_mean`: Example: `show_mean=False` hides the mean-return line; default=True draws a vertical line at the mean return.
- `sharey`: Example: `sharey=False` lets each panel use its own y-axis scale; default=True uses the same y-axis scale for all panels.
- `figsize`: Example: `figsize=(12, 3.5)` sets the figure width and height; default=None chooses the figure size automatically.
- `title`: Example: `title='Monthly Return Distributions'` adds a figure-level title; default=None adds no title.
- `alpha`: Example: `alpha=0.5` makes the histogram bars more transparent; default=0.8.
- `edgecolor`: Example: `edgecolor='black'` uses black borders; default='white'.
- `ncols`: Example: `ncols=2` uses two panels per row; default=3.

**Outputs**

- `figure`: The Matplotlib Figure containing the histogram layout.
- `axes`: A list of Matplotlib Axes objects, with one visible panel for each selected portfolio.

The function returns both objects so you can control the chart after it is created: The `fig` output lets you save or adjust the entire figure. The `axes` output lets you customize individual histograms.

For example, after the function runs:

- `axes[0].set_title('Low-momentum returns')` changes the title of the first histogram.
- `axes[1].set_title('Low-momentum returns')` changes the title of the second histogram.
- `fig.savefig('returns.png')` saves the complete figure as an image.

In [ ]:
fig, axes = fp.plot_return_histograms(
    returns=portfolio_returns,
    columns=['Dec 1', 'Dec 5', 'Dec 10'],
    title='Monthly Returns by Momentum Portfolio'
)
plt.show()

## Plot cumulative wealth

If one dollar is invested at the beginning of the sample and all returns are reinvested, its value evolves according to

$$W_T = (1+R_1)(1+R_2)...(1+R_T).$$

We can plot the cumulative wealth of one dollar invested in each portfolio using the `fp.plot_cumulative_wealth()` function.

`plot_cumulative_wealth()`

**Required Inputs**

- `returns`: A Pandas DataFrame containing portfolio returns as decimal values.

**Optional inputs** (defaults)

- `columns` Example: `columns=['Dec 1', 'Dec 10']` plots only the low- and high-ranked portfolios; default=all columns.
- `start` Example: `start='2000-01-01'` begins the wealth calculation in January 2000; default=None uses all observations.
- `figsize` Example: `figsize=(10, 5)` sets the figure width and height; default=(8.0, 4.0).
- `title` Example: `title='Momentum Portfolio Wealth'` changes the plot title; default='Growth of One Dollar'.
- `xlabel` Example: `xlabel='Month'` changes the horizontal-axis label; default='Date'.
- `ylabel` Example: `ylabel='Portfolio Value'` changes the vertical-axis label; default='Dollars'.
- `legend_ncols` Example: `legend_ncols=2` places the legend in two columns; default=3.
- `legend_fontsize` Example: `legend_fontsize=10` increases the legend text size; default=8.
- `grid_alpha` Example: `grid_alpha=0.5` makes the grid more visible; default=0.3.

In [ ]:
selected_portfolios = ['Dec 1', 'Dec 10']

fig, ax = fp.plot_cumulative_wealth(
    returns=portfolio_returns,
    columns=selected_portfolios,
    title='Growth of One Dollar by Momentum Portfolio',
)

plt.show()

## Try it
Here are some additional exercises to practice the `load_ken_french_data()` function.

1. Filter the `available` DataFrame to display only strategies with daily data. Which strategies have published daily files?

2. Change the first example from `strategy='momentum'` to `strategy='size'` or `strategy='booktomarket'`. Compare the returned column names and dates.

3. Change `data_type='deciles'` to `data_type='quintiles'` for the book-to-market example. How does the number of portfolios change?

4. Change `weighting='equal'` to `weighting='value'` for `bm_portfolios`. How do the portfolio returns differ?

5. Change `portfolio=[1, 5]` to `portfolio='low'`, `portfolio='high'`, or `portfolio='all'`. What columns are returned in each case?

6. Add an `end_date` to the daily size request. How does restricting the sample period change the returned DataFrame?

7. Use `details=True` with another strategy. What information does the function provide about the source and portfolio construction?

8. Recalculate `summary_stats` using only `Dec 1`, `Dec 5`, and `Dec 10`. Pass the selected columns to `fm.summary_stats()` and keep `frequency='monthly'`.

9. Change the histogram from 35 bins to 15 and then 75 bins. How does the choice of bins affect your interpretation of the return distribution?

10. Change `selected_portfolios` in the cumulative-wealth example to `['Dec 1', 'Dec 5', 'Dec 10']`. How do the growth paths compare?

11. Try to request daily size quintiles using `data_type='quintiles'` and `frequency='daily'`. Read the error message and explain why the request is unavailable.
12. Try to request daily data for `strategy='accruals'`. Use the error message to identify the frequencies published for that strategy.
13. Try `data_type='quintiles'` with `strategy='momentum'`. Explain why the loader rejects the request instead of creating quintiles by averaging deciles.